# Laboratorio 04 â€” Pipeline Completa Bronze â†’ Silver â†’ Gold con Notificaciones

**Semana:** 05 | **Actividad de referencia:** Actividad 04  
**Modalidad:** Individual | **Entorno:** Databricks Lakeflow (Spark Declarative Pipelines)

---

## Instrucciones generales

Integra todo lo aprendido en la semana para construir una pipeline Medallion completa (Bronze â†’ Silver â†’ Gold) con tu dataset propio, distribuida en mÃºltiples notebooks (uno por capa), con expectativas de calidad, parÃ¡metros, y notificaciones por webhook al completar o fallar.

> Este notebook es el **notebook Gold** y el orquestador del diseÃ±o. Los notebooks Bronze y Silver deben crearse por separado.

## Parte 1 â€” DescripciÃ³n del dataset y arquitectura de la pipeline

1. **Nombre, fuente y URL** del dataset.
2. **Arquitectura de la pipeline:** Â¿CuÃ¡ntos notebooks usarÃ¡ la pipeline? Â¿QuÃ© genera cada uno?
3. **KPIs Gold:** Â¿QuÃ© mÃ©tricas de negocio calcularÃ¡ la capa Gold?
4. **Webhook de notificaciÃ³n:** Â¿A quÃ© sistema notificarÃ¡s? (Slack, Teams, email via Zapier, etc.)
5. **Preguntas de negocio finales:** Al menos 3 que respondan los datos de la capa Gold.

**Arquitectura de tu pipeline:**

```
Notebook 01 â€” Bronze:
  @dp.table bronze_mi_dataset â€” Auto Loader desde /Volumes/.../landing/

Notebook 02 â€” Silver:
  @dp.table silver_mi_dataset â€” Limpieza + expectativas
  @dp.table quarantine_mi_dataset â€” Registros rechazados

Notebook 03 â€” Gold (este): 
  @dp.materialized_view gold_kpi_1 â€” Primer KPI de negocio
  @dp.materialized_view gold_kpi_2 â€” Segundo KPI de negocio
  @dp.table gold_final â€” Tabla Gold consolidada
```

**Escribe aquÃ­ tu arquitectura real:**

## Parte 2 â€” Importaciones y parÃ¡metros de pipeline

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# ParÃ¡metros de pipeline â€” all configurable from Lakeflow UI
ENTORNO           = spark.conf.get("entorno",           "dev")
RUTA_LANDING      = spark.conf.get("ruta_landing",      "/Volumes/workspace/default/week_5/landing/")
SCHEMA_LOCATION   = spark.conf.get("schema_location",   "/Volumes/workspace/default/week_5/schema/")
WEBHOOK_URL       = spark.conf.get("webhook_url",        "")  # rellena en Lakeflow Settings
UMBRAL_NULOS      = float(spark.conf.get("umbral_nulos", "30"))  # % mÃ¡ximo nulos aceptado

print(f"entorno        = {ENTORNO}")
print(f"ruta_landing   = {RUTA_LANDING}")
print(f"webhook_url    = {'configurado' if WEBHOOK_URL else 'NO configurado'")

## Parte 3 â€” Perfil tÃ©cnico del dataset (exploraciÃ³n interactiva)

In [ ]:
# ExploraciÃ³n antes de definir la pipeline
df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{RUTA_LANDING}/*.csv")

total = df.count()
print(f"Dataset: {total:,} filas | {len(df.columns)} columnas")
df.printSchema()
df.describe().show(truncate=False)

In [ ]:
# Nulos por columna
nulo_exprs = [
    F.round(
        F.sum(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), 1).otherwise(0))
        * 100.0 / total, 1
    ).alias(f"{c}")
    for c in df.columns
]
print("Porcentaje de nulos por columna:")
df.select(nulo_exprs).show(truncate=False)

In [ ]:
# DistribuciÃ³n de la columna de particiÃ³n Gold
df.groupBy("columna_particion_gold").count().orderBy(F.col("count").desc()).show(20)

**Decisiones de diseÃ±o basadas en el perfil:**  
1. Â¿QuÃ© columnas necesitan expectativas de calidad?
2. Â¿QuÃ© columna usarÃ¡s como particiÃ³n en Gold?
3. Â¿QuÃ© KPIs tienen sentido con la distribuciÃ³n que ves?

## Parte 4 â€” DefiniciÃ³n de la pipeline completa (Gold notebook)

Este notebook solo define la capa Gold. Los notebooks Bronze y Silver deben estar en `/semana_05/laboratorios/` y listarse en la configuraciÃ³n de la pipeline Lakeflow.

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Gold KPI 1: resumen de negocio principal
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.materialized_view(
    comment="Gold KPI 1: resumen agregado por categorÃ­a principal."
)
def gold_kpi_resumen():
    return (
        dp.read("silver_mi_dataset_calidad")  # nombre de la tabla Silver de lab_02
        .groupBy("columna_particion_gold")
        .agg(
            F.count("*").alias("total_registros"),
            F.avg("columna_numerica").alias("promedio"),
            F.sum("columna_numerica").alias("suma_total"),
            F.max("columna_numerica").alias("maximo"),
            F.min("columna_numerica").alias("minimo")
        )
        .withColumn("_gold_ts", F.current_timestamp())
        .withColumn("_entorno", F.lit(ENTORNO))
    )

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Gold KPI 2: ranking dentro de cada grupo
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from pyspark.sql.window import Window

@dp.materialized_view(
    comment="Gold KPI 2: top 10 registros por categorÃ­a."
)
def gold_kpi_top10_por_categoria():
    w = Window.partitionBy("columna_particion_gold").orderBy(F.col("columna_numerica").desc())
    return (
        dp.read("silver_mi_dataset_calidad")
        .withColumn("ranking", F.rank().over(w))
        .filter(F.col("ranking") <= 10)
        .orderBy("columna_particion_gold", "ranking")
    )

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Gold Final: tabla consolidada con todos los KPIs
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.table(
    comment="Gold final: tabla principal para consumo por BI/dashboards."
)
def gold_final_mi_dataset():
    df_resumen = dp.read("gold_kpi_resumen")
    # AÃ±ade mÃ¡s joins con otras vistas Gold si las tienes
    return df_resumen

## Parte 5 â€” ConfiguraciÃ³n de la pipeline multi-notebook y webhooks (teÃ³rico)

Describe aquÃ­ la configuraciÃ³n JSON de la pipeline completa:

```json
{
  "name": "lab05-04-pipeline-completa-mi-dataset",
  "target": "workspace.default",
  "clusters": [{"num_workers": 2}],
  "libraries": [
    {"notebook": {"path": "/semana_05/laboratorios/lab_02_autoloader_quality"}},
    {"notebook": {"path": "/semana_05/laboratorios/lab_04_pipeline_completa"}}
  ],
  "configuration": {
    "entorno":         "dev",
    "ruta_landing":    "/Volumes/workspace/default/week_5/landing/",
    "schema_location": "/Volumes/workspace/default/week_5/schema/",
    "webhook_url":     "https://hooks.slack.com/services/...",
    "umbral_nulos":    "30"
  },
  "notifications": [
    {
      "email_recipients": ["tu-email@dominio.com"],
      "alerts": ["on-update-failure", "on-flow-failure"]
    }
  ],
  "mode": "TRIGGERED"
}
```

**Edita el JSON arriba con tus valores reales. Luego responde:**
1. Â¿Por quÃ© conviene listar los notebooks en orden Bronze â†’ Silver â†’ Gold?
2. Â¿Lakeflow infiere las dependencias entre notebooks automÃ¡ticamente? Â¿CÃ³mo?
3. Â¿QuÃ© diferencia hay entre `on-update-failure` y `on-flow-failure`?

## Parte 6 â€” NotificaciÃ³n vÃ­a webhook (simulaciÃ³n)

In [ ]:
import requests
import json

def notificar_webhook(webhook_url: str, mensaje: str, estado: str = "success") -> None:
    """EnvÃ­a una notificaciÃ³n al webhook configurado. Solo ejecuta si hay URL."""
    if not webhook_url:
        print(f"[SIM] Webhook no configurado â€” mensaje: {mensaje}")
        return
    payload = {
        "text": f":{'white_check_mark' if estado == 'success' else 'x'}: *Pipeline Lab05-04* â€” {mensaje}"
    }
    try:
        resp = requests.post(webhook_url, json=payload, timeout=5)
        print(f"Webhook: {resp.status_code}")
    except Exception as e:
        print(f"Error en webhook: {e}")

# Simular notificaciÃ³n de Ã©xito
notificar_webhook(WEBHOOK_URL, "Pipeline completa finalizada correctamente.", "success")

## Parte 7 â€” Verificar resultados (notebook interactivo posterior)

In [ ]:
# Ejecutar DESPUÃ‰S de correr la pipeline completa en Lakeflow
tablas_pipeline = [
    "workspace.default.bronze_autoloader_mi_dataset",
    "workspace.default.silver_mi_dataset_calidad",
    "workspace.default.quarantine_mi_dataset",
    "workspace.default.gold_kpi_resumen",
    "workspace.default.gold_kpi_top10_por_categoria",
    "workspace.default.gold_final_mi_dataset",
]

print(f"{'Tabla':<50} {'Filas':>10}")
print("-" * 62)
for t in tablas_pipeline:
    try:
        cnt = spark.table(t).count()
        print(f"{t.split('.')[-1]:<50} {cnt:>10,}")
    except Exception as e:
        print(f"{t.split('.')[-1]:<50} ERROR")

In [ ]:
# Mostrar Gold KPI final
spark.table("workspace.default.gold_final_mi_dataset").orderBy(F.col("total_registros").desc()).show(15, truncate=False)

## Parte 8 â€” Preguntas de negocio sobre la capa Gold

Responde las 3 preguntas planteadas en la Parte 1 usando la tabla Gold.

In [ ]:
# Pregunta 1:
spark.sql("""
    SELECT * FROM workspace.default.gold_final_mi_dataset
    -- aÃ±ade tu condiciÃ³n
    LIMIT 10
""").show(truncate=False)

**ConclusiÃ³n pregunta 1:**

In [ ]:
# Pregunta 2:
spark.sql("""
    SELECT * FROM workspace.default.gold_kpi_top10_por_categoria
    WHERE ranking = 1
""").show(truncate=False)

**ConclusiÃ³n pregunta 2:**

In [ ]:
# Pregunta 3: libre â€” la mÃ¡s Ãºtil para un stakeholder
spark.sql("""

""").show(truncate=False)

**ConclusiÃ³n pregunta 3:**

## Parte 9 â€” ReflexiÃ³n final de la semana

1. Â¿QuÃ© diferencia fundamental hay entre un Job de Databricks (semana 04) y una Pipeline Lakeflow (semana 05)?
2. Â¿QuÃ© ventajas ofrece la pipeline declarativa sobre un script imperativo para reproducibilidad y mantenimiento?
3. Â¿CuÃ¡ndo elegirÃ­a el patrÃ³n CDC + SCD2 sobre una pipeline batch normal con `overwrite`?
4. Â¿QuÃ© harÃ­as diferente ahora que tienes experiencia con las 5 semanas si tuvieras que diseÃ±ar este pipeline desde cero?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_05/laboratorios/lab_04_pipeline_completa.ipynb semana_05/laboratorios/<tu-nombre>/lab_04_pipeline_completa.ipynb

git add semana_05/laboratorios/<tu-nombre>/lab_04_pipeline_completa.ipynb
git commit -m "lab: semana05 lab04 pipeline completa Bâ†’Sâ†’G webhook Gold KPIs <nombre-dataset> - <tu-nombre>"
git push origin develop
```